PhysicsBody.java

In [ ]:
import greenfoot.*; 

public abstract class PhysicsBody extends Actor
{
    private double exactX;
    private double exactY;


Ý Nghĩa : sử dụng từ khóa abstract vì lớp này không tạo đối tượng trực tiếp . Nó đóng vai trò là khung xương cho thực thể khác kế thừa 

In [ ]:

    @Override
    public void move(int distance)
    {
        move((double)distance);
    }
    
    public void move(double distance)
    {
        double radians = Math.toRadians(getRotation());
        double dx = Math.cos(radians) * distance;
        double dy = Math.sin(radians) * distance;
        setLocation(exactX + dx, exactY + dy);
    }
    
    


Cơ chế : Chuyển đổi từ tọa độ cực (góc và khoảng cách) sang tọa độ Descartes(x,y).
Toán học: Sử dụng hàm sin và cos để tính toán độ dời chính xác theo mọi góc độ. Điều này cực kì quan trọng
cho các loại đạn bay xiên hoặc hiệu ứng đẩy lùi

In [ ]:
public void setLocation(double x, double y) 
    {
        exactX = x;
        exactY = y;
        super.setLocation((int) (x + 0.5), (int) (y + 0.5));
    }

 @Override
    public void setLocation(int x, int y) 
    {
        exactX = x;
        exactY = y;
        super.setLocation(x, y);
    }


Chức năng: Đây là "trái tim" của cas. Nó động bộ hóa giữa Vị trí Logic(số thực mượt mà ) và Vị trí Hiển thị(pixel trên màn hình)

In [ ]:

    public double getExactX() 
    {
        return exactX;
    }

    public double getExactY() 
    {
        return exactY;
    }
}

Chức năng : Cung cấp tọa độ chính xác cho các class khác ( ví dụ: Bullet cần lấy vị trí chính xác của Zombie để tính va chạm đến từng pixel

_________________________________________________________________________________________________________________________

SpriteAnimator.java

In [ ]:
import greenfoot.*; 
import java.util.Timer;
import java.io.*;

public class SpriteAnimator extends PhysicsBody
{
    
    public long deltaTime; // khoảng thời gian chênh lệch giữa lần cập nhập ảnh này so với trước đó
    public long lastFrame = System.nanoTime(); // lưu lại thời điểm chính xác
    public long currentFrame = System.nanoTime(); // lưu thời gian hiện tại
    public GreenfootImage[] previousSprites = null; // Một mảng lưu trữ bộ ảnh mà nhân vật mới dùng ở lần chạy trước
    public int frame = 0;
 

In [ ]:
 public GreenfootImage[] importSprites(String filename, int frames) {
        GreenfootImage[] temp = new GreenfootImage[frames];
        for (int i = 0; i < frames; i++) {
            GreenfootImage scale = new GreenfootImage(filename+(i+1)+".png");
            
            temp[i] = scale;
        }
        return temp;
    }
    
    public GreenfootImage[] importSprites(String filename, int frames, double scaleFactor) {
        GreenfootImage[] temp = new GreenfootImage[frames];
        for (int i = 0; i < frames; i++) {
            GreenfootImage scale = new GreenfootImage(filename+(i+1)+".png");
            scale.scale((int)(scale.getWidth()*scaleFactor), (int)(scale.getHeight()*scaleFactor));
            temp[i] = scale;
        }
        return temp;
    }

1. Tham số truyền vào:
   - String filename: Cái tên gốc của file ảnh "Run","Attack"
   - int frames: Tổng số tấm ảnh có trong bộ hoạt ảnh đó
   - double scaleFactor : Tỷ lệ phóng to/ thu nhỏ (trong hàm thứ 2).
2. vòng lặp chạy từ 0 đến số frame có , trả về cái khay đã đầy ảnh
3. ở dòng thứ 2 nếu nhập scaleFactor=2.0, Ảnh gốc rộng 100px, thì sẽ tính ra 200px và phóng to tấm ảnh đó lên gấp đôi ngay khi vừa nạp vào game.

In [ ]:
public void animate(GreenfootImage[] sprite, long duration) {
        currentFrame = System.nanoTime();
        deltaTime = (currentFrame - lastFrame) / 1000000;
        
        
        if (deltaTime > duration) {
            lastFrame = currentFrame;    
            frame++;
            if (frame < sprite.length) {
                setImage(sprite[frame]);
            }

        }
        if (frame < sprite.length) {
            
        } else {
            
            frame = 0;
            setImage(sprite[frame]);
        }
        if (!sprite.equals(previousSprites)) {
            setImage(sprite[frame]);
            previousSprites = sprite;
        }
        
           
    }
    
    public void animate(GreenfootImage[] sprite, long duration, boolean loop) {
        currentFrame = System.nanoTime();
        deltaTime = (currentFrame - lastFrame) / 1000000;
           
        if (deltaTime > duration) {
            
            lastFrame = currentFrame;    
            frame++;
            if (frame < sprite.length) {
                setImage(sprite[frame]);
            }
        }
        if (frame < sprite.length) {
            
        } else if (loop) {
            
            frame = 0;
            setImage(sprite[frame]);
        } else {
            
        }
        if (!sprite.equals(previousSprites)) {
            if (frame < sprite.length) {
                setImage(sprite[frame]);
                previousSprites = sprite;
            }
        }  
    }

currentFrame = System.nanoTime();
deltaTime = (currentFrame - lastFrame) / 1000000;

Logic: Máy tính chạy rất nhanh , nếu chỉ viết frame++ , ảnh sẽ biến đổi nhanh đến mức mắt người không thấy gì.
Phân tích: Lấy thời gian hiện tại trừ đi mốc thời gian cũ . Kết quả chia cho 1 triệu ra đơn vị mili giây(ms). Đây là khoảng thời gian thực tế đã trôi qua giữa 2 lần máy tính vẽ lại màn hình 

if (deltaTime > duration) {
    lastFrame = currentFrame;    
    frame++;
    if (frame < sprite.length) {
        setImage(sprite[frame]);
    }
}

Logic: duration chính là "tốc độ thay hình". Ví dụ để duration = 100 , tức là cứ mỗi 100ms thì mới cho phép ảnh đổi một lần 

Phân tích: "Nếu deltaTime chưa đủ lớn (chưa đến 100ms), nó bỏ qua hết , không tăng frame
- Nếu đủ rồi : nó câp nhập lại lastFrame (coi như bắt đầu chu kỳ mới ), tăng chỉ số frame lên 1 và ngay lập tức hình ảnh mới cho nhân vật bằng setImage

In [ ]:
   
   
    
    
    
    
    public void setFrame(int toFrame) {
        frame = toFrame-1;
    
    }
    
    public int getCurrentFrame() {
        return frame+1;
    }
    public void hitFlash(GreenfootImage[] sprite, String filename) {
        int tempFrame = frame;
        int flashFrame = tempFrame+1;
        if (tempFrame >= sprite.length) {
            tempFrame = 0;
            flashFrame = 1;
          
        } else if (flashFrame >= sprite.length) {
            flashFrame = 0;
        }
      
        
        
        GreenfootImage first = new GreenfootImage("flash"+filename+(tempFrame+1)+".png");
        GreenfootImage second = new GreenfootImage("flash"+filename+(flashFrame+1)+".png");
 
        sprite[tempFrame] = first;
        sprite[flashFrame] = second;
        
        
        setImage(sprite[tempFrame]);
        
        Timer timer = new Timer();
        timer.schedule(new Timer1(sprite, filename, tempFrame,flashFrame), 500L); 
       
    }

   
    

}
